# import libraries

In [4]:
import os
import pandas as pd
import sqlite3

# Create Project Folder Structure

In [3]:
# Create folders
folders = ["raw", "processed", "output"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully!")


Folders created successfully!


# load raw dataset

In [6]:
# Load dataset
file_path = "customer_churn_dataset-testing-master.csv"
df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)
print(df.head())

Dataset Loaded Successfully!
Shape: (64374, 12)
   CustomerID  Age  Gender  Tenure  Usage Frequency  Support Calls  \
0           1   22  Female      25               14              4   
1           2   41  Female      28               28              7   
2           3   47    Male      27               10              2   
3           4   35    Male       9               12              5   
4           5   53  Female      58               24              9   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0             27             Basic         Monthly          598   
1             13          Standard         Monthly          584   
2             29           Premium          Annual          757   
3             17           Premium       Quarterly          232   
4              2          Standard          Annual          533   

   Last Interaction  Churn  
0                 9      1  
1                20      0  
2                21      0  
3           

In [7]:
df.to_csv("raw/customer_churn_raw.csv", index=False)

In [8]:
df

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0
...,...,...,...,...,...,...,...,...,...,...,...,...
64369,64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64370,64371,37,Male,6,1,5,22,Standard,Annual,923,9,1
64371,64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1
64372,64373,50,Female,18,19,7,22,Standard,Monthly,540,13,1


# data clening (Missing values + Duplicates)

In [9]:
# Check missing values
print("Missing values:\n", df.isnull().sum())

# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates
before_duplicates = df.shape[0]
df = df.drop_duplicates()
after_duplicates = df.shape[0]

print("Duplicates removed:", before_duplicates - after_duplicates)


Missing values:
 CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64
Duplicates removed: 0


C:\Users\Bergith Shafro\AppData\Local\Temp\ipykernel_19512\3980342169.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\Bergith Shafro\AppData\Local\Temp\ipykernel_19512\3980342169.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

# standardize  column Name & Datatype

In [10]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Updated Columns:\n", df.columns)

# Convert specific columns if needed (example)
# df['tenure'] = df['tenure'].astype(int)
# df['monthlycharges'] = df['monthlycharges'].astype(float)


Updated Columns:
 Index(['customerid', 'age', 'gender', 'tenure', 'usage_frequency',
       'support_calls', 'payment_delay', 'subscription_type',
       'contract_length', 'total_spend', 'last_interaction', 'churn'],
      dtype='object')


# creat derived columns

In [11]:
# Example margin column (if revenue & cost exist)
if 'monthlycharges' in df.columns:
    df['estimated_margin'] = df['monthlycharges'] * 0.2  # assume 20% margin

# Segment flag example
if 'tenure' in df.columns:
    df['loyal_customer_flag'] = df['tenure'].apply(lambda x: 1 if x > 24 else 0)

# Churn binary flag
if 'churn' in df.columns:
    df['churn_flag'] = df['churn'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)


In [12]:
df.to_csv("processed/customer_churn_cleaned.csv", index=False)

In [13]:
df

,customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn,loyal_customer_flag,churn_flag
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1,1,0
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0,1,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0,1,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0,0,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64369,64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1,1,0
64370,64371,37,Male,6,1,5,22,Standard,Annual,923,9,1,0,0
64371,64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1,1,0
64372,64373,50,Female,18,19,7,22,Standard,Monthly,540,13,1,0,0


# split into separate output

# customers table

In [14]:
customers_cols = ['customerid', 'gender', 'seniorcitizen', 
                  'partner', 'dependents', 'tenure']

customers = df[[col for col in customers_cols if col in df.columns]]
customers.to_csv("output/customers.csv", index=False)


# sevices/orders Table

In [15]:
orders_cols = ['customerid', 'phoneservice', 'internetservice', 
               'contract', 'paymentmethod', 'monthlycharges']

orders = df[[col for col in orders_cols if col in df.columns]]
orders.to_csv("output/orders.csv", index=False)

# products/Billing table

In [16]:
products_cols = ['customerid', 'totalcharges', 'estimated_margin', 'churn_flag']

products = df[[col for col in products_cols if col in df.columns]]
products.to_csv("output/products.csv", index=False)


# losd into sqllite dataset

In [17]:
conn = sqlite3.connect("output/customer_churn.db")

customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

print("Data loaded into SQLite successfully!")


Data loaded into SQLite successfully!


# validate counts before& after

In [18]:
print("Original count:", pd.read_csv("raw/customer_churn_raw.csv").shape[0])
print("After cleaning:", df.shape[0])
print("Customers table:", customers.shape[0])
print("Orders table:", orders.shape[0])
print("Products table:", products.shape[0])


Original count: 64374
After cleaning: 64374
Customers table: 64374
Orders table: 64374
Products table: 64374


# Readme File 